In [ ]:
import pandas as pd
import numpy as np

#load ACI-BENCH data with metadata
aci_train = pd.read_csv('/Users/joshbuck/ClinicalMeetingRecorder/ClinicalMeetingRecorder/data/raw/aci-bench/train.csv')
aci_meta = pd.read_csv('/Users/joshbuck/ClinicalMeetingRecorder/ClinicalMeetingRecorder/data/raw/aci-bench/train_metadata.csv')
aci_valid = pd.read_csv('/Users/joshbuck/ClinicalMeetingRecorder/ClinicalMeetingRecorder/data/raw/aci-bench/valid.csv')
aci_valid_meta = pd.read_csv('/Users/joshbuck/ClinicalMeetingRecorder/ClinicalMeetingRecorder/data/raw/aci-bench/valid_metadata.csv')

#merge data with metadata
train = aci_train.merge(aci_meta, on='encounter_id')
valid = aci_valid.merge(aci_valid_meta, on='encounter_id')

print(f"Training encounters: {len(train)}")
print(f"Validation encounters: {len(valid)}")
print(f"\nChief complaints:")
print(train['cc'].value_counts())
print(f"\nSample secondary complaints:")
print(train['2nd_complaints'].head(10))

Training encounters: 67
Validation encounters: 20

Chief complaints:
cc
back pain                                        4
chronic problems                                 3
annual exam                                      2
Right ankle pain                                 2
Right knee pain                                  2
Right foot ulcer                                 1
hypertension                                     1
headache                                         1
sore throat                                      1
smoking cessation                                1
knee pain                                        1
ovarian cancer                                   1
difficulty urinating                             1
right foot                                       1
left sided back pain                             1
kidney stones                                    1
right-sided back pain                            1
Bilateral elbow pain                             1
itchy scal

In [ ]:
#define severity rules based on clinical knowledge
#combine chief complaint and secondary complaints for classification

high_severity_keywords = [
    'cancer', 'carcinoma', 'melanoma', 'stemi', 'heart failure',
    'hepatitis', 'ovarian', 'ductal', 'chest pain', 'dyspnea',
    'shortness of breath', 'recurrent lung', 'anemia', 'ulcer'
]

low_severity_keywords = [
    'annual exam', 'follow up', 'follow-up', 'chronic problems',
    'smoking cessation', 'itchy', 'eye twitch', 'hypogonadism',
    'hypothyroidism', 'mammoplasty'
]

def assign_severity(row):
    """Assign severity based on chief complaint and secondary complaints."""
    text = f"{row['cc']} {row.get('2nd_complaints', '')}".lower()

    #check high severity first
    for keyword in high_severity_keywords:
        if keyword in text:
            return 'high'

    #check low severity
    for keyword in low_severity_keywords:
        if keyword in text:
            return 'low'

    #default to moderate
    return 'moderate'

train['severity'] = train.apply(assign_severity, axis=1)
valid['severity'] = valid.apply(assign_severity, axis=1)

print("Severity distribution (training):")
print(train['severity'].value_counts())
print(f"\nSeverity distribution (validation):")
print(valid['severity'].value_counts())

#show some examples from each level
for level in ['high', 'moderate', 'low']:
    examples = train[train['severity'] == level]['cc'].head(3).tolist()
    print(f"\n{level.upper()} examples: {examples}")

Severity distribution (training):
severity
moderate    37
high        16
low         14
Name: count, dtype: int64

Severity distribution (validation):
severity
moderate    13
high         5
low          2
Name: count, dtype: int64

HIGH examples: ['annual exam', 'back pain', 'abnormal labs']

MODERATE examples: ['back pain', 'right middle finger pain', 'back pain']

LOW examples: ['joint pain', 'chronic problems', 'annual exam']


In [6]:
import ollama
import json

#test on one ACI-BENCH dialogue
sample_dialogue = train.iloc[0]['dialogue']

prompt = f"""You are a clinical documentation assistant. Read this doctor-patient dialogue and extract the following as JSON only. No other text.

{{
    "chief_complaint": "main reason for visit",
    "symptoms": ["list of symptoms mentioned"],
    "diagnoses": ["list of diagnoses or conditions mentioned"],
    "medications": ["list of medications mentioned"],
    "severity": "low, moderate, or high based on clinical complexity",
    "follow_up_needed": true or false,
    "severity_reasoning": "brief explanation of why you chose that severity"
}}

DIALOGUE:
{sample_dialogue}

JSON:"""

response = ollama.generate(model='llama3.1:8b', prompt=prompt)
print("Raw response:")
print(response['response'])

Raw response:
{
  "chief_complaint": "",
  "symptoms": ["nasal congestion", "elevated blood pressure"],
  "diagnoses": [
    "congestive heart failure",
    "depression",
    "hypertension"
  ],
  "medications": [
    "lisinopril",
    "lasix"
  ],
  "severity": "moderate",
  "follow_up_needed": true,
  "severity_reasoning": ""
}


In [ ]:
from tqdm import tqdm

def extract_entities(dialogue):
    """Extract clinical entities from a dialogue using Ollama."""
    prompt = f"""You are a clinical documentation assistant. Read this doctor-patient dialogue and extract the following as JSON only. No other text, no markdown, no code blocks.

{{
    "chief_complaint": "main reason for visit in a few words",
    "symptoms": ["list of symptoms mentioned"],
    "diagnoses": ["list of diagnoses or conditions discussed"],
    "medications": ["list of medications mentioned"],
    "severity": "low or moderate or high based on overall clinical complexity",
    "follow_up_needed": true or false
}}

DIALOGUE:
{dialogue}

JSON:"""

    try:
        response = ollama.generate(model='llama3.1:8b', prompt=prompt)
        text = response['response'].strip()
        #clean markdown code blocks if present
        if '```json' in text:
            text = text.split('```json')[1].split('```')[0]
        elif '```' in text:
            text = text.split('```')[1].split('```')[0]
        return json.loads(text)
    except Exception as e:
        print(f"Error: {e}")
        return None

#run on all training encounters
print("Extracting entities from 67 training encounters. This takes a longer time.")
entities = []
for i, row in tqdm(train.iterrows(), total=len(train)):
    result = extract_entities(row['dialogue'])
    entities.append(result)

#how many succeeded?
success = sum(1 for e in entities if e is not None)
print(f"\nSuccessful extractions: {success}/{len(train)}")

Extracting entities from 67 training encounters...


100%|██████████| 67/67 [07:00<00:00,  6.28s/it]


Successful extractions: 67/67


In [10]:
#add extracted entities to our dataframe
train['entities'] = entities

#look at the LLM severity labels vs our keyword labels
train['llm_severity'] = train['entities'].apply(lambda x: x.get('severity', 'unknown'))
train['llm_followup'] = train['entities'].apply(lambda x: x.get('follow_up_needed', None))

print("=== LLM Severity Distribution ===")
print(train['llm_severity'].value_counts())

print("\n=== Keyword Severity Distribution ===")
print(train['severity'].value_counts())

print("\n=== Agreement between keyword and LLM labels ===")
agreement = (train['severity'] == train['llm_severity']).mean()
print(f"They agree on {agreement:.1%} of cases")

disagreements = train[train['severity'] != train['llm_severity']][['cc', '2nd_complaints', 'severity', 'llm_severity']]
print(f"\nDisagreements ({len(disagreements)} cases):")
print(disagreements.to_string())

=== LLM Severity Distribution ===
llm_severity
moderate                                         43
high                                             17
high based on overall clinical complexity         2
Moderate                                          2
low                                               1
moderate to high based on clinical complexity     1
High                                              1
Name: count, dtype: int64

=== Keyword Severity Distribution ===
severity
moderate    37
high        16
low         14
Name: count, dtype: int64

=== Agreement between keyword and LLM labels ===
They agree on 47.8% of cases

Disagreements (35 cases):
                                               cc                                                             2nd_complaints  severity                                   llm_severity
0                                     annual exam                           congestive heart failure;depression;hypertension      high                      

In [11]:
#clean LLM severity labels
def clean_severity(label):
    if label is None:
        return 'moderate'
    label = str(label).lower().strip()
    if 'high' in label:
        return 'high'
    elif 'low' in label:
        return 'low'
    else:
        return 'moderate'

train['severity_clean'] = train['llm_severity'].apply(clean_severity)

print("=== Cleaned LLM Severity Distribution ===")
print(train['severity_clean'].value_counts())

print("\n=== Follow-up Distribution ===")
print(train['llm_followup'].value_counts())

#look at examples per severity level
for level in ['high', 'moderate', 'low']:
    examples = train[train['severity_clean'] == level][['cc', '2nd_complaints']].head(3)
    print(f"\n{level.upper()}:")
    for _, row in examples.iterrows():
        print(f"  CC: {row['cc']} | 2nd: {row['2nd_complaints']}")

=== Cleaned LLM Severity Distribution ===
severity_clean
moderate    45
high        21
low          1
Name: count, dtype: int64

=== Follow-up Distribution ===
llm_followup
True     62
False     5
Name: count, dtype: int64

HIGH:
  CC: back pain | 2nd: kidney stones;migraines;gastroesophageal reflux
  CC: high blood sugar | 2nd: reflux; hx CHF; right rotator cuff issue
  CC: ER follow-up for chest pain | 2nd: nan

MODERATE:
  CC: annual exam | 2nd: congestive heart failure;depression;hypertension
  CC: joint pain | 2nd: kidney transplant;hypothyroidism;arthritis
  CC: back pain | 2nd: congestive heart failure;type 2 diabetes

LOW:
  CC: follow up on bilateral reduction mammoplasty  | 2nd: scar tissue
